# QR_Intersect(lhs_geom, rhs_geom, cell)

Check if qr_envp intersect on the lower left corner intersection.

Here we are looking for the intersection of 2 dataframes with polygons.

In [23]:
import geofunctions as S
import pyspark.sql.functions as F

In [24]:
cell = 50.0

In [25]:
lhs = [("POLYGON((0 0,0 100,100 100,100 0,0 0))",)]

ldf = (
    spark.createDataFrame(lhs, "wkt string")
    .select(S.st_fromtext("wkt").alias("lgeom"))
    .select("lgeom", F.explode(S.qr_envp("lgeom", cell, dist=0.0)).alias("lqr"))
)

In [26]:
rhs = [("POLYGON((75 75,75 200,200 200,200 75,75 75))",)]

rdf = (
    spark.createDataFrame(rhs, "wkt string")
    .select(S.st_fromtext("wkt").alias("rgeom"))
    .select("rgeom", F.explode(S.qr_envp("rgeom", cell, dist=0.0)).alias("rqr"))
)

In [28]:
(ldf
    .join(rdf, ldf.lqr.qr == rdf.rqr.qr)
     .filter(S.qr_intersect("lqr", "rqr", cell))
     .select(
         S.st_astext(S.st_intersection("lgeom", "rgeom")).alias("intersection")
     )
     .show(truncate=False)
)

+--------------------------------------------------------+
|intersection                                            |
+--------------------------------------------------------+
|MULTIPOLYGON (((75 75, 100 75, 100 100, 75 100, 75 75)))|
+--------------------------------------------------------+

